In [0]:
import pyspark
from pyspark.sql.types import *
import pyspark.sql.functions as f
from pyspark.sql import SparkSession, Row, DataFrame
from pyspark.sql.window import Window
from pyspark.sql.functions import col
from pyspark.sql.functions import when
from typing import Union, Optional, List
from dataclasses import dataclass
from pyspark.sql.types import IntegerType
from functools import reduce
from datetime import timedelta
from pyspark.sql.functions import broadcast
from pyspark.sql.functions import when, lit
from pyspark.sql.functions import collect_set, lpad

#import
#from pls_common_data_store import pls_data_store
#pds = pls_data_store()

#ignore strange depreciation warnings
from warnings import simplefilter 
simplefilter(action='ignore', category=DeprecationWarning)
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
spark.conf.set("spark.sql.shuffle.partitions","auto")
spark.conf.get("spark.sql.shuffle.partitions")

# spark.conf.set("spark.databricks.queryWatchdog.maxQueryTasks", "50000000")

In [0]:
# Pre-pivoted closed loop data pulled from closed_loop_campaign_summary notebook
closed_loop_prepivot = spark.read.option("header", "true").csv('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/closed_loop_summary_tab_INTERMEDIATE.csv')
closed_loop_prepivot = closed_loop_prepivot.filter(f.col('camp_start_date') >= '2023-01-01')
closed_loop_prepivot.display()

# Metadata pulls from KPM
mmci = spark.read.parquet(f'abfss://data@sa8451midsrprd.dfs.core.windows.net/media_meas_campaign_info_v')
mda = spark.read.parquet(f'abfss://measure@sa8451camprd.dfs.core.windows.net/dashboard/campaign/version=v2/source=azure')
points_detail = (spark.read.parquet(f'abfss://data@sa8451kemprd.dfs.core.windows.net/pls_points_v2/'))
mmoi= spark.read.parquet(f'abfss://landingzone@sa8451entlakegrnprd.dfs.core.windows.net/mart/comms/prd/measurement/MEDIA_MEAS_OFFER_INFO')
new_th = spark.read.parquet(f'abfss://landingzone@sa8451entlakegrnprd.dfs.core.windows.net/mart/comms/prd/measurement/TARGET_HISTORY')
old_th = spark.read.parquet(f'abfss://landingzone@sa8451entlakegrnprd.dfs.core.windows.net/mart/comms/prd/measurement/bullseye/TARGET_HISTORY_FULL_20251015/').withColumnRenamed('ehhn', 'hshd_code').select('TARGET_ID', 'HSHD_CODE', 'PRIORITY', 'TEST_CONTROL_ID', 'OFFER_ID', 'DECILE', 'SCORE')
target_history = new_th.union(old_th)
mhtv = spark.read.parquet(f'abfss://data@sa8451midsrprd.dfs.core.windows.net/media_hist_revamped')
redemptions = spark.read.parquet(f'abfss://acds@sa8451posprd.dfs.core.windows.net/transaction_coupon_fct')
downloads = spark.read.parquet(f'abfss://measure@sa8451camprd.dfs.core.windows.net/intermediate/engagements/coupon_downloads/')
# Rem sse push barcodes automated
rem_sse_push_offer_codes = spark.read.parquet(f'abfss://tmkrprtnrs-users@sa8451krprtnrdev.dfs.core.windows.net/a136627/META_DATA_INFO')

#### REM SSE PUSH: Absolute Metric Calculation

In [0]:
# TO-DO: Only run new campaigns: that is, campaigns that exist in media history that are not inside below path
abs_rem_sse_push_xcm = spark.read.parquet(f'abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_rem_sse_push_xcm')
abs_rem_sse_push_xcm.display()

In [0]:
# Handle media history error case. campaign id 138141 XCM EMOD PUSH REM should be type XCM
closed_loop_prepivot = closed_loop_prepivot.withColumn(
    "campaign_type",
    when(f.col("campaign_id") == 138141, lit("XCM")).otherwise(f.col("campaign_type"))
)

closed_loop_prepivot.display()
#closed_loop_prepivot.filter(f.col("campaign_id") == 138141).display()

In [0]:
# 6/19/2026: 2026 REM SSE PUSH offer codes
rem_sse_push_offer_codes.display()
rem_sse_push_offer_codes_small = rem_sse_push_offer_codes.filter(
    f.col("CHANNEL").isin(["Single Subject Email", "Push Notifications"])
).select("COUPON_BARCODE", "KPM_DUPLICATED_ID", "KPM_PROJECT_ID", "CHANNEL")
# Make into list of tuples, similar to offsite 
barcode_id_camptype_tuple = [(row[0], row[1], row[2], row[3]) for row in rem_sse_push_offer_codes_small.collect()]
barcode_id_camptype_tuple

In [0]:
# 6/19/2026: Turn tuple of list into dataframe
schema = ["coupon_barcode", "duplicated_id", "campaign_id", "campaign_type"]
coupon_df = spark.createDataFrame(barcode_id_camptype_tuple, schema)
coupon_df = coupon_df.withColumnRenamed("campaign_id", "campaign_id_new")
coupon_df.display()

campaign_ids = coupon_df.select('campaign_id_new').distinct().rdd.flatMap(lambda x: x).collect()
print(campaign_ids)

In [0]:
# only pull new rem sse push.. if its updated, output should be empty
abs_rem_sse_push_xcm = spark.read.parquet('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_rem_sse_push_xcm')
existing_campaign_ids = [row['campaign_id'] for row in abs_rem_sse_push_xcm.select('campaign_id').distinct().collect()]

print(existing_campaign_ids)
# These campaigns have dummy UPC or no data in mmoi
exclude_campaign_ids = [79065, 146580, 71365]

closed_loop_prepivot_rem_sse_push = closed_loop_prepivot.filter(
    (
        f.col("project_name").contains("XCM SSE PUSH REM") |
        f.col("project_name").contains("XCM EMOD PUSH REM")
    ) &
    (~f.col("project_name").contains("Credit")) &
    (~f.col("project_name").contains("Kroger Pay")) &
    (~f.col("campaign_id").isin(existing_campaign_ids)) &
    (~f.col("campaign_id").isin(exclude_campaign_ids))
)

closed_loop_prepivot_rem_sse_push = closed_loop_prepivot_rem_sse_push.filter(
    f.year(f.col("year")).isin([2026, 2027])
)

#campaign_ids = [row['campaign_id'] for row in closed_loop_prepivot_rem_sse_push.select('campaign_id').distinct().collect()]
closed_loop_prepivot_rem_sse_push.display()

In [0]:
closed_loop_prepivot_rem_sse_push_small = (
    closed_loop_prepivot_rem_sse_push
    .join(
        coupon_df,
        (closed_loop_prepivot_rem_sse_push.campaign_id == coupon_df.duplicated_id) &
        (closed_loop_prepivot_rem_sse_push.channel == coupon_df.campaign_type),
        how="inner"
    )
    .dropDuplicates()
)

#closed_loop_prepivot_rem_sse_push_small.select("campaign_id", "channel", "camp_start_date", "camp_end_date").display()
# 6/19/2026: 2026 REM SSE PUSH offer codes
closed_loop_prepivot_rem_sse_push_select = closed_loop_prepivot_rem_sse_push_small.select("campaign_id_new", "channel", "camp_start_date", "camp_end_date", "camp_cost", "coupon_barcode")
closed_loop_prepivot_rem_sse_push_select.display()

In [0]:
# Working Cost: Camp Cost * Multiplier (depending on channel type. Made changes on 6/19 to reflect single channel
mmci_sse_rem_push_working_cost = (closed_loop_prepivot_rem_sse_push_select.filter(f.col('campaign_id_new').isin(campaign_ids))
    .withColumn('multiplier',
        f.when(f.col('channel') == 'Display Ad', f.lit(0.3520))
         .when(f.col('channel') == 'Email Module', f.lit(0.015))
         .when(f.col('channel') == 'Pandora', f.lit(0.741))
         .when(f.col('channel') == 'Pinterest', f.lit(0.663))
         .when(f.col('channel') == 'Pre-Roll Video', f.lit(.3960))
         .when(f.col('channel') == 'Push Notifications', f.lit(0.038))
         .when(f.col('channel') == 'Roku', f.lit(0.90))
         .when(f.col('channel') == 'Native', f.lit(1))
         .when(f.col('channel') == 'Single Subject Email', f.lit(0.3390))
         .when(f.col('channel') == 'Targeted Digital Coupon', f.lit(0.141))
         .otherwise(f.lit(0))
    )
    .withColumn('working_cost', (f.col('camp_cost') * f.col('multiplier'))) # Keep as double for now
    .groupBy('campaign_id_new', 'channel')
    .agg(
        f.max('CAMP_START_DATE').alias('camp_start_date'),
        f.max('CAMP_END_DATE').alias('camp_end_date'),
        f.sum('camp_cost').alias('camp_cost_mmci'),
        f.sum('working_cost').alias('working_cost'),
    )
    .withColumnRenamed('campaign_id_new', 'campaign_id')
)

mmci_sse_rem_push_working_cost.display()

In [0]:
# 6/19 made a change to not bring in barcodes from mmoi, since its causing measurement issues. bring directly from tracker
mmoi_metadata_no_barcode = (mmoi
    .withColumn('EFFECTIVE_DATE', f.date_format(f.to_date('EFFECTIVE_DATE', 'yyyy-MM-dd'), 'yyyyMMdd'))
    .withColumn('EXPIRATION_DATE', f.date_format(f.to_date('EXPIRATION_DATE', 'yyyy-MM-dd'), 'yyyyMMdd'))
    #.withColumn('redemption_barcode', f.lpad(f.col('COUPON_BARCODE').cast('string'), 13, '0'))
    .filter(f.col('KPM_PROJECT_ID').isin(campaign_ids))
    .groupBy('KPM_PROJECT_ID')
    .agg(
        #f.collect_set('COUPON_BARCODE').alias('coupon_barcodes'),
        #f.collect_set('redemption_barcode').alias('redemption_barcodes'),
        f.first('EFFECTIVE_DATE').alias('effective_date'),
        f.first('EXPIRATION_DATE').alias('expiration_date')
    )
)

mmoi_metadata_no_barcode.display()

In [0]:
# 6/19: Pull Barcodes NOT through mmoi, but through array
closed_loop_prepivot_rem_sse_push_barcodes = (
    closed_loop_prepivot_rem_sse_push_select
    .withColumn("coupon_barcode_padded", lpad(f.col("coupon_barcode").cast("string"), 12, "0"))
    .groupBy("campaign_id_new")
    .agg(collect_set("coupon_barcode_padded").alias("coupon_barcodes"))
)

mmoi_metadata = closed_loop_prepivot_rem_sse_push_barcodes.join(
    mmoi_metadata_no_barcode,
    closed_loop_prepivot_rem_sse_push_barcodes.campaign_id_new == mmoi_metadata_no_barcode.KPM_PROJECT_ID,
    how="inner"
).select("KPM_PROJECT_ID", "effective_date","expiration_date", "coupon_barcodes")

display(mmoi_metadata)

In [0]:
# Calculate working cost, iroas (sales uplift / campaign cost), aroas (sales test total / campaign cost)
mhtv_metrics_agg_rem_sse_push = (closed_loop_prepivot_rem_sse_push_small
    .groupBy('campaign_id_new')
    .agg(
        f.avg('sales_uplift_total').alias('sales_uplift_total'),
        f.avg('sales_test_total').alias('sales_test_total'),
        f.avg('camp_cost').alias('camp_cost')
    )
    #.withColumn('working_cost', f.col('camp_cost') * 0.3390)
    .withColumn('iroas_original', f.round(f.col('sales_uplift_total') / f.col('camp_cost'), 2))
    .withColumn('aroas_original', f.round(f.col('sales_test_total') / f.col('camp_cost'), 2))
    .withColumnRenamed('campaign_id_new', 'campaign_id')
)

mhtv_metrics_agg_rem_sse_push.display()

In [0]:
# Join offer/redemption barcode data with original ROAS + uplift #s with campaign info
mmoi_mhtv_rem_sse_push = (mmoi_metadata
    .join(mhtv_metrics_agg_rem_sse_push, mmoi_metadata["KPM_PROJECT_ID"] == mhtv_metrics_agg_rem_sse_push["campaign_id"], how="right")
)

mmoi_mhtv_rem_sse_push = (mmoi_mhtv_rem_sse_push
    .join(mmci_sse_rem_push_working_cost.select("campaign_id", "working_cost"), on = "campaign_id", how = "inner")
)

mmoi_mhtv_rem_sse_push = (mmoi_mhtv_rem_sse_push
    .join(mmci.select('kpm_project_id', 'target_id', 'project_name'), on = "KPM_PROJECT_ID", how = "inner")
)

mmoi_mhtv_rem_sse_push.display()

In [0]:
target_ids_to_keep = [
    row['target_id'] for row in mmoi_mhtv_rem_sse_push.select('target_id').distinct().collect()
]

# Reduce Target history table with only relevant target ids
test_hhs_slim = (
    target_history
    .filter(f.col('test_control_id') == '1')
    .filter(f.col('target_id').isin(target_ids_to_keep)) 
    .select(f.col('HSHD_CODE').alias('ehhn'), 'target_id')
    .distinct()
)

# Join Target history with metadata
th_filter = (
    test_hhs_slim.join(
        f.broadcast(mmoi_mhtv_rem_sse_push.select('kpm_project_id', 'project_name', 'target_id').distinct()), 
        on='target_id', 
        how='inner'
    )
)

th_filter.display()

In [0]:
# Join target history + metadata with points detail
points_detail = (spark.read.parquet(f'abfss://data@sa8451kemprd.dfs.core.windows.net/pls_points_v2/'))
points_detail_target_history = (broadcast(th_filter)
    .join(points_detail, on='ehhn', how='inner'))
display(points_detail_target_history)

In [0]:
'''
# Select only columns needed from mmoi_mhtv_rem_sse_push
mmoi_mhtv_rem_sse_push_filtered = (mmoi_mhtv_rem_sse_push
    .select(
        'kpm_project_id', 
        'coupon_barcodes', 
        #'redemption_barcodes', 
        'effective_date',
        'expiration_date'
    )
)

# Convert transaction date to datetype 
points_detail_target_history = points_detail_target_history.withColumn(
    'trn_dt', f.to_date('trn_dt', 'yyyyMMdd')
)
points_detail_target_history.display()
'''

In [0]:
# Select only columns needed from mmoi_mhtv_rem_sse_push
mmoi_mhtv_rem_sse_push_filtered = (mmoi_mhtv_rem_sse_push
    .select(
        'kpm_project_id', 
        'coupon_barcodes', 
        #'redemption_barcodes', 
        'effective_date',
        'expiration_date'
    )
)

# Convert transaction date to datetype 
points_detail_target_history = points_detail_target_history.withColumn(
    'trn_dt', f.to_date('trn_dt', 'yyyyMMdd')
)

# Join points detail + target history with dashboard and metadata (iroas, aroas, barcodes, camp info)
points_detail_target_history_offer_info = (
    points_detail_target_history
    .join(
        f.broadcast(mmoi_mhtv_rem_sse_push_filtered),
        on=[
            points_detail_target_history.kpm_project_id == mmoi_mhtv_rem_sse_push_filtered.kpm_project_id,
            (f.array_contains(mmoi_mhtv_rem_sse_push_filtered.coupon_barcodes, points_detail_target_history.offer))],
            #f.array_contains(mmoi_mhtv_rem_sse_push_filtered.redemption_barcodes, points_detail_target_history.offer))
            #points_detail_target_history.trn_dt.between(mmoi_mhtv_mmci_sse_filtered.effective_date, mmoi_mhtv_mmci_sse_filtered.expiration_date)
        how="inner"
    ).drop(mmoi_mhtv_rem_sse_push_filtered.kpm_project_id)
)

points_detail_target_history_offer_info.display()

In [0]:
# Convert effective_date and expiration_date to date type, then filter to only transaction date is between effective and expiry date
points_detail_target_history_filtered_offer_info = (
    points_detail_target_history_offer_info
    .withColumn('effective_date', f.to_date('effective_date', 'yyyyMMdd'))
    .withColumn('expiration_date', f.to_date('expiration_date', 'yyyyMMdd'))
    .filter(f.col('trn_dt').between(f.col('effective_date'), f.col('expiration_date')))
)

points_detail_target_history_filtered_offer_info.display()

In [0]:
#  Sum of points earned across all HHs per campaign
aggregated_df = (points_detail_target_history_filtered_offer_info
    .groupBy('kpm_project_id', 'project_name')
    .agg(
        f.count('ehhn').alias('total_hhs'), 
        f.countDistinct('ehhn').alias('distinct_ehhn'), 
        f.sum('points_earned').alias('total_points_earned')
    )
)

aggregated_df.display()

In [0]:
# Calculations (from Shumaila's code)
final_result_rem_sse_push_df = (mmoi_mhtv_rem_sse_push
    .join(aggregated_df, on=['kpm_project_id', 'project_name'], how='left')
    .withColumn('cost_total_points_earned', f.round(f.col('total_points_earned') * 0.01, 2).cast('double'))
    .withColumn('cost_total_points_redemeed', f.round(f.col('total_points_earned') * 0.016 * 0.5887, 2).cast('double'))
    .withColumn('adj_sales_uplift', f.col("sales_uplift_total").cast('Integer'))
    .withColumn('adj_sales_total', f.col("sales_test_total").cast('Integer'))
    .withColumn('camp_cost', f.round(f.col("camp_cost").cast('double'), 2))
    .withColumn('working_cost', f.round(f.col("working_cost").cast('double'), 2))
    .withColumn('adj_total_cost', f.round((f.col('cost_total_points_earned') + f.col('camp_cost')).cast('double'), 2))
    .withColumn('abs_total_cost', f.round((f.col('cost_total_points_earned') + f.col('working_cost')).cast('double'), 2))
    # Redemption Cost
    .withColumn('redemption_cost', f.col('abs_total_cost') - f.col('working_cost'))
    .withColumn('abs_sales_uplift_earned', f.round((f.col('adj_sales_uplift') - f.col('redemption_cost')).cast('double'), 2))
    .withColumn('abs_sales_test_earned', f.round((f.col('adj_sales_total') - f.col('redemption_cost')).cast('double'), 2))
    .withColumn('adj_iroas', f.round((f.col('adj_sales_uplift') / f.col('adj_total_cost')).cast('double'), 2))
    .withColumn('abs_iroas', f.round((f.col('abs_sales_uplift_earned') / f.col('abs_total_cost')).cast('double'), 2))
    .withColumn('adj_aroas', f.round((f.col('adj_sales_total') / f.col('adj_total_cost')).cast('double'), 2))
    .withColumn('abs_aroas', f.round((f.col('abs_sales_test_earned') / f.col('abs_total_cost')).cast('double'), 2))
)

# Display the final aggregated DataFrame
final_result_rem_sse_push_df.display()

In [0]:
# Dropping array columns (barcodes) to support csv and parquet writing
final_result_rem_sse_push_barcodes_dropped = final_result_rem_sse_push_df.drop('coupon_barcodes')
final_result_rem_sse_push_barcodes_dropped.coalesce(1).write.mode("append").parquet('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_rem_sse_push_xcm')